# Unit 7 - 01 - Train a conditional diffusion model

We train a small **conditional DDPM**: the UNet receives the noisy high-res image
**and** the low-res image (concatenated -> 2 input channels) and learns to predict the
added noise. At sampling time we start from pure noise and denoise it while keeping the
LR image as conditioning - turning the LR scan into a high-res estimate.

This is according [SR3](https://arxiv.org/abs/2104.07636)


In [ ]:
import os, csv, random
import numpy as np
import tifffile
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from skimage.transform import resize
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

torch.manual_seed(0); random.seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


In [ ]:
# --- config ---
MARKER     = "CELL2"
MANIFEST   = "./prepared/pairs_x4_%s.csv" % MARKER
IMG_SIZE   = 128          # working resolution. At 128 the LR is used natively; HR is downscaled to 128.
BATCH_SIZE = 16           # fits < 16 GB at IMG_SIZE=128. Lower it if you raise IMG_SIZE.
EPOCHS     = 30
LR         = 1e-4
TIMESTEPS  = 1000
CKPT_DIR   = "./checkpoints"
CKPT       = os.path.join(CKPT_DIR, "sr3_caco2_x4.pt")
os.makedirs(CKPT_DIR, exist_ok=True)

# At IMG_SIZE=512 (native HR) you get full-resolution SR but need a bigger GPU / smaller batch.

In [ ]:
class SRCaco2Pairs(Dataset):
    """Returns (lr, hr) tensors in [-1, 1], shape (1, IMG_SIZE, IMG_SIZE)."""
    def __init__(self, manifest, split, img_size, augment=False):
        with open(manifest) as f:
            self.rows = [r for r in csv.DictReader(f) if r["split"] == split]
        self.img_size, self.augment = img_size, augment

    def __len__(self):
        return len(self.rows)

    @staticmethod
    def _load(path):
        img = tifffile.imread(path)
        if img.ndim == 3:
            img = img[..., 0] if img.shape[-1] <= 4 else img[0]
        return np.asarray(img, dtype=np.float32)

    def _prep(self, img):
        if img.shape[0] != self.img_size:
            img = resize(img, (self.img_size, self.img_size), order=3,
                         preserve_range=True, anti_aliasing=True)
        return img / 127.5 - 1.0          # 8-bit [0,255] -> [-1, 1]

    def __getitem__(self, i):
        r = self.rows[i]
        lr = self._prep(self._load(r["lr_path"]))
        hr = self._prep(self._load(r["hr_path"]))
        if self.augment:
            if random.random() < 0.5: lr, hr = lr[:, ::-1].copy(), hr[:, ::-1].copy()
            if random.random() < 0.5: lr, hr = lr[::-1].copy(),    hr[::-1].copy()
        lr = torch.from_numpy(lr)[None].float()
        hr = torch.from_numpy(hr)[None].float()
        return lr, hr

train_ds = SRCaco2Pairs(MANIFEST, "train", IMG_SIZE, augment=True)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=4, drop_last=True, pin_memory=True)
print("train pairs:", len(train_ds))


In [ ]:
model = UNet2DModel(
    sample_size=IMG_SIZE,
    in_channels=2,                 # noisy HR (1) + LR conditioning (1)
    out_channels=1,                # predicted noise (single channel)
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 256),
    down_block_types=("DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
).to(device)

scheduler = DDPMScheduler(num_train_timesteps=TIMESTEPS)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print("UNet parameters: %.1f M" % n_params)


In [ ]:
use_amp = (device == "cuda")
model.train()
for epoch in range(EPOCHS):
    running = 0.0
    pbar = tqdm(train_dl, desc="epoch %d/%d" % (epoch + 1, EPOCHS))
    for lr_img, hr_img in pbar:
        lr_img, hr_img = lr_img.to(device), hr_img.to(device)
        noise = torch.randn_like(hr_img)
        t = torch.randint(0, TIMESTEPS, (hr_img.size(0),), device=device).long()
        noisy_hr = scheduler.add_noise(hr_img, noise, t)
        model_in = torch.cat([noisy_hr, lr_img], dim=1)          # (B, 2, H, W)

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=use_amp):
            pred = model(model_in, t).sample
            loss = F.mse_loss(pred, noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        running += loss.item()
        pbar.set_postfix(loss="%.4f" % loss.item())
    print("epoch %d  mean loss %.4f" % (epoch + 1, running / len(train_dl)))

if device == "cuda":
    print("peak GPU memory: %.2f GB" % (torch.cuda.max_memory_allocated() / 1e9))


In [ ]:
@torch.no_grad()
def sample(model, lr_img, num_steps=50):
    """DDIM-sample a high-res estimate conditioned on lr_img (in [-1,1])."""
    ddim = DDIMScheduler(num_train_timesteps=TIMESTEPS)
    ddim.set_timesteps(num_steps)
    model.eval()
    x = torch.randn_like(lr_img)
    for t in ddim.timesteps:
        noise_pred = model(torch.cat([x, lr_img], dim=1), t).sample
        x = ddim.step(noise_pred, t, x).prev_sample
    return x.clamp(-1, 1)

# quick visual preview on a few training pairs
lr_b, hr_b = next(iter(train_dl))
lr_b, hr_b = lr_b[:4].to(device), hr_b[:4]
gen = sample(model, lr_b).cpu()

def to01(x): return ((x + 1) / 2).clamp(0, 1)
fig, ax = plt.subplots(4, 3, figsize=(8, 10))
for i in range(4):
    ax[i, 0].imshow(to01(lr_b[i, 0]).cpu(), cmap="gray"); ax[i, 0].set_title("LR (input)")
    ax[i, 1].imshow(to01(gen[i, 0]),        cmap="gray"); ax[i, 1].set_title("generated")
    ax[i, 2].imshow(to01(hr_b[i, 0]),       cmap="gray"); ax[i, 2].set_title("HR (target)")
    for a in ax[i]: a.axis("off")
plt.tight_layout(); plt.show()


In [ ]:
torch.save({"state_dict": model.state_dict(),
            "img_size": IMG_SIZE, "timesteps": TIMESTEPS, "marker": MARKER}, CKPT)
print("saved", CKPT)

Checkpoint saved. Continue to **`02_Evaluate.ipynb`** for the SSIM evaluation.
